In [1]:
from tensorzinb.tensorzinb import TensorZINB
import numpy as np
import time

from scipy.stats import uniform, binom, nbinom, bernoulli
import statsmodels.api as sm

## Poisson initialization validation for Negative Binomial

In [2]:
np.random.seed(1)                 # set seed to replicate example
nobs= 25000                          # number of obs in model 

xb = np.ones((nobs,1)) *0.7
theta = 0.5

exb = np.exp(xb)
nby = nbinom.rvs(exb, theta)
X=np.ones((nobs,1))

In [3]:
zinbo= TensorZINB(nby.reshape((-1,1)),X)

In [4]:
inits = zinbo._poisson_init()
inits

{'x_mu': array([[0.69494556]]), 'theta': array([[0.69711051]])}

In [5]:
r=zinbo.fit()
r

I0000 00:00:1773683944.195331  180704 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


{'llf_total': np.float64(-47019.8343944344),
 'llfs': array([-47019.83439443]),
 'aic_total': np.float64(94043.6687888688),
 'aics': array([94043.66878887]),
 'df_model_total': np.int64(2),
 'df': np.int64(2),
 'weights': {'x_mu': array([[0.69489694]], dtype=float32),
  'theta': array([[0.6957012]], dtype=float32)},
 'cpu_time': 0.11649417877197266,
 'num_sample': 25000,
 'epochs': 51}

## Poisson initialization validation for Zero-inflated Negative Binomial 

In [6]:
np.random.seed(1)                    # set seed to replicate example
nobs= 25000                          # number of obs in model 

xb = np.ones((nobs,1)) *0.7
theta = 0.5

xc = 2.0
exc = 1.0 / (1.0 + np.exp(-xc))

p = bernoulli.rvs(exc, size=(nobs,1))

exb = np.exp(xb)
nby = nbinom.rvs(exb, theta)*p
X=np.ones((nobs,1))

In [7]:
zinbo= TensorZINB(nby.reshape((-1,1)),X,exog_infl=X)

In [8]:
inits = zinbo._poisson_init()
inits

{'x_mu': array([[0.57050486]]),
 'x_pi': array([[-3.52493127]]),
 'theta': array([[0.33882783]])}

In [9]:
r=zinbo.fit()
r

{'llf_total': np.float64(-45113.591992329886),
 'llfs': array([-45113.59199233]),
 'aic_total': np.float64(90233.18398465977),
 'aics': array([90233.18398466]),
 'df_model_total': np.int64(3),
 'df': np.int64(3),
 'weights': {'x_mu': array([[0.6893516]], dtype=float32),
  'x_pi': array([[-2.074867]], dtype=float32),
  'theta': array([[0.6561708]], dtype=float32)},
 'cpu_time': 1.3920860290527344,
 'num_sample': 25000,
 'epochs': 860}

## Negative Binomial

In [10]:
np.random.seed(1)                 # set seed to replicate example
nobs= 25000                          # number of obs in model 

x1 = binom.rvs(1, 0.6, size=nobs)   # categorical explanatory variable
x2 = uniform.rvs(size=nobs)         # real explanatory variable

theta = 0.5
X = sm.add_constant(np.column_stack((x1, x2)))
beta = [1.0, 2.0, -1.5]
xb = np.dot(X, beta)          # linear predictor

exb = np.exp(xb)
nby = nbinom.rvs(exb, theta)

In [11]:
zinbo= TensorZINB(nby.reshape((-1,1)),X)

In [12]:
r=zinbo.fit()
r

{'llf_total': np.float64(-59241.247901022376),
 'llfs': array([-59241.24790102]),
 'aic_total': np.float64(118490.49580204475),
 'aics': array([118490.49580204]),
 'df_model_total': np.int64(4),
 'df': np.int64(4),
 'weights': {'x_mu': array([[ 1.0032264],
         [ 1.9956939],
         [-1.497616 ]], dtype=float32),
  'theta': array([[2.1339564]], dtype=float32)},
 'cpu_time': 0.36870884895324707,
 'num_sample': 25000,
 'epochs': 277}

this is the definition of dispersion in statsmodels

In [13]:
np.exp(-r['weights']['theta'])

array([[0.11836805]], dtype=float32)

In [14]:
start_time = time.time()
mod = sm.NegativeBinomial(nby, X).fit(maxiter=100, disp=False, warn_convergence=False)
cpu_time = time.time() - start_time
mod.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     NegativeBinomial Regression Results                      
==============================================================================
Dep. Variable:                      y   No. Observations:                25000
Model:               NegativeBinomial   Df Residuals:                    24997
Method:                           MLE   Df Model:                            2
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                  0.2064
Time:                        13:59:39   Log-Likelihood:                -59241.
converged:                       True   LL-Null:                       -74651.
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0028      0.010     96.502      0.000       0.982       1.023
x1             1.9953      0.010    200.495      0.000       1.976       2.015
x2            -1.4980      0.013   -118.673      0.000      -1.523      -1.473
alpha          0.1184      0.003     42.072      0.000       0.113       0.124
==============================================================================
"""

In [15]:
cpu_time

0.05975508689880371

TensorZINB matches statsmodels for Negative Binomial

## Zero-inflated Negative Binomial 

In [16]:
np.random.seed(1)                 # set seed to replicate example
nobs= 25000                          # number of obs in model 

x1 = binom.rvs(1, 0.6, size=nobs)   # categorical explanatory variable
x2 = uniform.rvs(size=nobs)         # real explanatory variable

theta = 0.5
X = sm.add_constant(np.column_stack((x1, x2)))
beta = [1.0, 2.0, -1.5]
xb = np.dot(X, beta)          # linear predictor

exb = np.exp(xb)

xc = 3.0
exc = 1.0 / (1.0 + np.exp(-xc))

p = bernoulli.rvs(exc, size=(nobs,1))

nby = nbinom.rvs(exb, theta).reshape((-1,1))*p
X_infl=np.ones((nobs,1))

In [17]:
zinbo= TensorZINB(nby.reshape((-1,1)),X,exog_infl=X_infl)

In [18]:
r=zinbo.fit()
r

{'llf_total': np.float64(-59281.92643573915),
 'llfs': array([-59281.92643574]),
 'aic_total': np.float64(118573.8528714783),
 'aics': array([118573.85287148]),
 'df_model_total': np.int64(5),
 'df': np.int64(5),
 'weights': {'x_mu': array([[ 1.0398031],
         [ 1.9678903],
         [-1.5097274]], dtype=float32),
  'x_pi': array([[-2.7315137]], dtype=float32),
  'theta': array([[2.196405]], dtype=float32)},
 'cpu_time': 0.5324099063873291,
 'num_sample': 25000,
 'epochs': 293}

this is the definition of dispersion in statsmodels

In [19]:
np.exp(-r['weights']['theta'])

array([[0.11120222]], dtype=float32)

statsmodels cannot solve

In [20]:
start_time = time.time()
mod = sm.ZeroInflatedNegativeBinomialP(nby.reshape((-1,1)),X,exog_infl=X_infl).fit()
cpu_time = time.time() - start_time
mod.summary()

/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3937: RuntimeWarning: invalid value encountered in log
  a1 * np.log(a1) + y * np.log(mu) -
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3938: RuntimeWarning: invalid value encountered in log
  (y + a1) * np.log(a2))
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3974: RuntimeWarning: invalid value encountered in log
  dgterm = dgpart + np.log(a1 / a2) + 1 - a3 / a2
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:4329: RuntimeWarning: overflow encountered in exp
  return np.exp(linpred)
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-pa

         Current function value: nan
         Iterations: 1
         Function evaluations: 112
         Gradient evaluations: 112


<class 'statsmodels.iolib.summary.Summary'>
"""
                     ZeroInflatedNegativeBinomialP Regression Results                    
=========================================================================================
Dep. Variable:                                 y   No. Observations:                25000
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                    24997
Method:                                      MLE   Df Model:                            2
Date:                           Mon, 16 Mar 2026   Pseudo R-squ.:                     nan
Time:                                   13:59:55   Log-Likelihood:                    nan
converged:                                 False   LL-Null:                       -72991.
Covariance Type:                       nonrobust   LLR p-value:                       nan
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
inflate_const  -351.6158        nan        nan        nan         nan         nan
const           213.8910        nan        nan        nan         nan         nan
x1              100.9036        nan        nan        nan         nan         nan
x2              101.6834        nan        nan        nan         nan         nan
alpha          -339.1602        nan        nan        nan         nan         nan
=================================================================================
"""

statsmodels cannot solve even with Poisson init

In [21]:
inits = zinbo._poisson_init()
start_params=np.concatenate([inits['x_pi'].flatten(),inits['x_mu'].flatten(),np.exp(-inits['theta'].flatten())])

In [22]:
start_time = time.time()
mod = sm.ZeroInflatedNegativeBinomialP(nby.reshape((-1,1)),X,exog_infl=X_infl).fit(start_params=start_params)
cpu_time = time.time() - start_time
mod.summary()

/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3937: RuntimeWarning: invalid value encountered in log
  a1 * np.log(a1) + y * np.log(mu) -
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3938: RuntimeWarning: invalid value encountered in log
  (y + a1) * np.log(a2))
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3974: RuntimeWarning: invalid value encountered in log
  dgterm = dgpart + np.log(a1 / a2) + 1 - a3 / a2
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3937: RuntimeWarning: divide by zero encountered in log
  a1 * np.log(a1) + y * np.log(mu) -
/opt/anaconda3/envs/tensorzinb-new/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:3937: RuntimeWarning: invalid value encountered in multiply
  a1 * np.log(a1) + y * np.log(mu) -
/opt/anaconda3/env

         Current function value: nan
         Iterations: 2
         Function evaluations: 113
         Gradient evaluations: 113


<class 'statsmodels.iolib.summary.Summary'>
"""
                     ZeroInflatedNegativeBinomialP Regression Results                    
=========================================================================================
Dep. Variable:                                 y   No. Observations:                25000
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                    24997
Method:                                      MLE   Df Model:                            2
Date:                           Mon, 16 Mar 2026   Pseudo R-squ.:                     nan
Time:                                   14:00:03   Log-Likelihood:                    nan
converged:                                 False   LL-Null:                       -72991.
Covariance Type:                       nonrobust   LLR p-value:                       nan
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
inflate_const     7.1325        nan        nan        nan         nan         nan
const           -78.1776        nan        nan        nan         nan         nan
x1              -88.9186        nan        nan        nan         nan         nan
x2              -47.4675        nan        nan        nan         nan         nan
alpha          -679.3743        nan        nan        nan         nan         nan
=================================================================================
"""

statsmodels only works when given optimized params generated by TensorZINB. It still takes very long time in this case.

In [23]:
inits = r['weights']
start_params=np.concatenate([inits['x_pi'].flatten(),inits['x_mu'].flatten(),np.exp(-inits['theta'].flatten())])

In [24]:
start_time = time.time()
mod = sm.ZeroInflatedNegativeBinomialP(nby.reshape((-1,1)),X,exog_infl=X_infl).fit(start_params=start_params)
cpu_time = time.time() - start_time
mod.summary()

Optimization terminated successfully.
         Current function value: 2.371275
         Iterations: 5
         Function evaluations: 8
         Gradient evaluations: 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                     ZeroInflatedNegativeBinomialP Regression Results                    
=========================================================================================
Dep. Variable:                                 y   No. Observations:                25000
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                    24997
Method:                                      MLE   Df Model:                            2
Date:                           Mon, 16 Mar 2026   Pseudo R-squ.:                  0.1878
Time:                                   14:00:09   Log-Likelihood:                -59282.
converged:                                  True   LL-Null:                       -72991.
Covariance Type:                       nonrobust   LLR p-value:                     0.000
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
inflate_const    -2.7314      0.038    -72.582      0.000      -2.805      -2.658
const             1.0393      0.011     91.869      0.000       1.017       1.061
x1                1.9675      0.011    181.047      0.000       1.946       1.989
x2               -1.5102      0.013   -116.512      0.000      -1.536      -1.485
alpha             0.1112      0.003     39.645      0.000       0.106       0.117
=================================================================================
"""

In [25]:
cpu_time

0.1608572006225586

TensorZINB matches statsmodels for Zero-inflated Negative Binomial when TensorZINB results are used for statsmodels initialization. The estimated weights from TensorZINB are close to true values used to generate the samples.

## with common features

In [26]:
import numpy as np
import statsmodels.api as sm

from scipy.stats import uniform, norm, bernoulli,poisson,nbinom


# Data
np.random.seed(141)                        # set seed to replicate example
nobs= 5000                                 # number of obs in model 
phi = 2

x1 = uniform.rvs(size=nobs)
xc0 = uniform.rvs(size=nobs)

xb = 1 + 2.0 * x1 + 1.5*xc0                        # linear predictor
xc = 2 - 3.0 * x1 + 0.8*xc0

exb = np.exp(xb)          
exc = 1.0 / (1.0 + np.exp(-xc))

p = bernoulli.rvs(exc)

nby = nbinom.rvs(phi, phi/(phi+exb))
zipy = 0*p+(1-p)*nby

X = np.transpose(x1)
X = sm.add_constant(X)

Y= np.reshape(zipy,(-1,1))

xb = 1.5 + 2.5 * x1+ 1.5*xc0                      # linear predictor
xc = 1 - 2.0 * x1 + 0.8*xc0

exb = np.exp(xb)          
exc = 1.0 / (1.0 + np.exp(-xc))

p = bernoulli.rvs(exc)

nby = nbinom.rvs(phi, phi/(phi+exb))
zipy = 0*p+(1-p)*nby

X = np.transpose(x1)
X = sm.add_constant(X)

Y0= np.reshape(zipy,(-1,1))

Y2=np.stack([Y,Y0],axis=1)[:,:,0]

X=X
X_infl=X
X_c=np.reshape(xc0,(-1,1))
X_infl_c=X_c

In [27]:
zinbo= TensorZINB(Y2,X,exog_c=X_c,exog_infl=X_infl,exog_infl_c=X_infl_c,same_dispersion=True)

In [28]:
r=zinbo.fit()
r

{'llf_total': np.float64(-21314.336705327107),
 'llfs': array([ -8837.55393428, -12476.78277104]),
 'aic_total': np.float64(42650.67341065421),
 'aics': array([17689.10786857, 24967.56554209]),
 'df_model_total': np.int64(11),
 'df': np.int64(7),
 'weights': {'theta': array([[0.7238577]], dtype=float32),
  'x_mu': array([[0.94033056, 1.4813366 ],
         [2.0251582 , 2.4956481 ]], dtype=float32),
  'z_mu': array([[1.553109]], dtype=float32),
  'x_pi': array([[ 2.097939 ,  1.0550491],
         [-3.0719712, -2.0062106]], dtype=float32),
  'z_pi': array([[0.7213149]], dtype=float32)},
 'cpu_time': 0.5850238800048828,
 'num_sample': 5000,
 'epochs': 395}